# 12 Causal Inference — Exercises

Practice causal inference concepts with the Songbai Nursing Home Legionella data.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statsmodels.formula.api as smf

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## Question 1: Attributable Risk of Hydrotherapy Exposure

1. Calculate the attack rate among hydrotherapy users (`hydrotherapy_use == 1`) and non-users
2. Calculate the AR (attributable risk) and PAR (population attributable risk)
3. Compare with the AR/PAR for shower exposure—which exposure contributes more?
4. Interpret from a causal inference perspective: what does AR represent? What assumptions does it require?

In [ ]:
# TODO: attack rate for hydrotherapy users vs non-users
# TODO: AR and PAR
# TODO: compare with shower exposure

## Question 2: Changing the DiD Intervention Date

1. Change the intervention date from 1/25 to 1/22
2. Keep the treated group as Wing B on floors 2-3, and the control group as the remaining areas
3. Build the panel data and run the DiD regression
4. How do the `treated:post` coefficient and p-value change?
5. Why does changing the intervention date affect the results?

In [ ]:
# TODO: change the intervention date to 1/22
# TODO: build the panel data
# TODO: DiD regression
# TODO: compare the results

## Question 3 (Challenge): Demonstrating Collider Bias

The DAG tells us that `hospitalized` is a collider (← severity, ← infection).
If you only analyze hospitalized patients, you create a spurious association.

1. Calculate the RR for shower_use → infected in the whole sample
2. Restrict to hospitalized patients only (`hospitalized == 1`) and calculate the RR again
3. Are the two RRs different? Why?
4. Explain the difference using the concept of a collider

In [ ]:
# TODO: RR for the whole sample
# TODO: RR restricted to hospitalized patients
# TODO: compare the difference
# TODO: collider interpretation

## Question 4: DiD Evaluation of a Vaccination Policy (Vaccination Policy Scenario)

Some districts rolled out a vaccination booster campaign; use difference-in-differences (DiD) to evaluate the policy's effect.

1. Compute DiD by hand from the 2×2 group means
2. Estimate DiD using the interaction term from `smf.ols("incidence ~ treated * post")` and compare
3. Interpret whether the interaction term is close to the true value, and explain how DiD removes the common time trend

In [ ]:
# Vaccination policy DiD: some districts rolled out a vaccination booster campaign (treated); compare incidence before and after the policy
rng = np.random.default_rng(1204)
_rows = []
TRUE_EFFECT = -8.0   # the policy truly reduces incidence by 8/100k
for dz in range(200):
    treated = 1 if dz < 100 else 0
    base = rng.normal(45, 6)
    for post in (0, 1):
        inc = base - 3 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 4)
        _rows.append({"district": dz, "treated": treated, "post": post, "incidence": inc})
vax = pd.DataFrame(_rows)
print(f"DiD data: {vax['district'].nunique()} districts × 2 periods, true effect = {TRUE_EFFECT}/100k")

# TODO: compute DiD by hand using the 2x2 group means = (treated post-pre) - (control post-pre)
# TODO: use smf.ols("incidence ~ treated * post", vax).fit() to get the interaction coefficient and compare with the hand calculation
# TODO: interpret whether the interaction coefficient is close to the true value -8, and explain why DiD removes the common time trend

## Question 5: DiD Evaluation of a Mask Mandate (Mask Mandate Scenario)

Some counties implemented a mask mandate; the outcome is the weekly case growth rate.

1. Estimate DiD using `smf.ols("growth ~ treated * post")`
2. Interpret the direction and magnitude of the mask mandate's causal effect

In [ ]:
# Mask mandate DiD: some counties implemented a mask mandate (treated); outcome is the weekly case growth rate
rng = np.random.default_rng(1205)
_rows = []
TRUE_EFFECT = -0.18   # the mask mandate reduces the growth rate by 0.18
for reg in range(160):
    treated = 1 if reg < 80 else 0
    base = rng.normal(0.30, 0.05)
    for post in (0, 1):
        g = base - 0.05 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 0.04)
        _rows.append({"region": reg, "treated": treated, "post": post, "growth": g})
mask = pd.DataFrame(_rows)
print(f"DiD data: {mask['region'].nunique()} counties × 2 periods, true effect = {TRUE_EFFECT}")

# TODO: use smf.ols("growth ~ treated * post", mask) to estimate DiD (the interaction term)
# TODO: interpret the direction and magnitude of the mask mandate's causal effect

## Question 6: Smoking and Disease—Confounding by Age (Smoking Scenario)

Age is a common cause of smoking and disease (a confounder).

1. Compute the crude OR (`smf.logit("disease ~ smoke")`)
2. Compute the adjusted OR (adding `age`)
3. Compare the smoke coefficients between the crude and adjusted models, and interpret the direction of age's confounding

In [ ]:
# Smoking and disease: age is a confounder (older people smoke more often and are also more likely to get sick)
rng = np.random.default_rng(1206)
n = 3000
age = rng.integers(20, 80, n)
smoke = rng.binomial(1, 1 / (1 + np.exp(-(-2.5 + 0.05 * age))))
logit = -4.5 + 0.05 * age + 0.8 * smoke     # smoke's true log-OR = 0.8
disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
dat = pd.DataFrame({"age": age, "smoke": smoke, "disease": disease})
print(f"n={n}, smoking rate={smoke.mean():.1%}, disease rate={disease.mean():.1%} (smoke's true log-OR=0.8)")

# TODO: compute the crude OR (smoke vs disease only, using smf.logit("disease ~ smoke"))
# TODO: compute the adjusted OR (adding age: smf.logit("disease ~ smoke + age"))
# TODO: compare the smoke coefficients between the crude and adjusted models, and interpret the direction of age's confounding

## Question 7: Propensity-Score Matching for COVID-19 Treatment (COVID-19 Scenario)

Sicker patients are more likely to be treated (confounding by indication); use propensity-score matching to estimate the treatment effect.

1. First look at why the naive death-rate difference is biased
2. Estimate the propensity score using logistic regression (severity → treated)
3. Use `NearestNeighbors` to match each treated case to its nearest control
4. Compute the matched ATT and compare with the true value −0.15

In [ ]:
# Propensity-score matching for COVID-19 treatment: sicker patients are more likely to be treated (confounding), but treatment is actually beneficial
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
rng = np.random.default_rng(1207)
n = 2000
severity = rng.uniform(0, 1, n)
treated = rng.binomial(1, 0.15 + 0.7 * severity)  # sicker patients are treated more often (confounding by indication; overlap is preserved)
TRUE_EFFECT = -0.15                              # treatment truly reduces the death rate by 0.15
death_p = (0.10 + 0.75 * severity + TRUE_EFFECT * treated).clip(0.01, 0.99)
death = rng.binomial(1, death_p)
cov = pd.DataFrame({"severity": severity, "treated": treated, "death": death})
print(f"n={n}, treated proportion={treated.mean():.1%}, true treatment effect (death-rate difference)={TRUE_EFFECT}")

# TODO: first compute the naive death-rate difference (treated - control), and see why it's biased (may even look harmful)
# TODO: estimate the propensity score using LogisticRegression(severity → treated)
# TODO: for each treated case, use NearestNeighbors to find the control with the closest propensity score
# TODO: compute the average death-rate difference after matching (the ATT), and compare with the true value -0.15

## Question 8 (Challenge): Instrumental Variables / 2SLS (Causal Identification Scenario)

The exposure is affected by an unobserved confounder U (endogenous); use an instrumental variable Z with two-stage least squares.

1. Naive OLS: `smf.ols("outcome ~ exposure")` → see how it is overestimated due to U
2. 2SLS: first stage `exposure ~ Z`, take the fitted values, second stage `outcome ~ exp_hat`
3. Compare the naive and IV coefficients, explain why IV recovers the true value 1.5, and what three conditions Z must satisfy

In [ ]:
# Instrumental variables IV: the exposure is affected by an unobserved confounder U (endogenous); Z is the instrumental variable (Challenge)
rng = np.random.default_rng(1208)
n = 3000
U = rng.normal(0, 1, n)                 # unobserved confounder
Z = rng.normal(0, 1, n)                 # instrumental variable: affects exposure, does not directly affect the outcome
exposure = 0.6 * Z + 0.7 * U + rng.normal(0, 1, n)
TRUE_EFFECT = 1.5
outcome = TRUE_EFFECT * exposure + 1.2 * U + rng.normal(0, 1, n)
iv = pd.DataFrame({"Z": Z, "exposure": exposure, "outcome": outcome})
print(f"n={n}, true causal effect of exposure = {TRUE_EFFECT} (naive OLS will overestimate this due to U)")

# TODO: naive OLS: smf.ols("outcome ~ exposure", iv) → see how the coefficient is overestimated due to U
# TODO: two-stage least squares 2SLS:
#        stage1 = smf.ols("exposure ~ Z", iv).fit(); iv["exp_hat"]=stage1.fittedvalues
#        stage2 = smf.ols("outcome ~ exp_hat", iv).fit()
# TODO: compare the naive and IV coefficients, explain why IV recovers the true value 1.5, and what conditions Z must satisfy

## Question 9: Food Poisoning AR/PAR and DiD for a Sanitation Intervention

A suspected food poisoning outbreak occurred in an elementary school's lunch program (this problem's data is a synthetic teaching scenario, not a real case). Health authorities suspect the "cold cucumber salad" served that day as the likely contamination source; after an audit, the catering company was required to strengthen its sanitation and disinfection measures.

**Part 1: AR / PAR (Attributable Risk)**

1. Use the `a, b, c, d` values from the 2×2 table below to compute the attack rates for the "ate the cold cucumber salad" and "did not eat it" groups
2. Compute the risk ratio (RR) and attributable risk (AR, risk difference)
3. Use the Levin formula to compute the population attributable fraction (PAF): `Pe * (RR - 1) / (1 + Pe * (RR - 1))`, where Pe is the proportion of everyone who ate that dish
4. Interpret the PAF: if this dish were removed from the menu, what proportion of cases could theoretically be prevented?

**Part 2: DiD (Effect of the Sanitation Measures)**

5. Using the panel data below, estimate the intervention effect of the sanitation measures with `smf.ols("cases ~ treated + post + treated:post", data=food).fit(cov_type="HC3")` (HC3-robust standard errors)
6. What does the `treated:post` interaction represent? How does it compare with the true effect built into the data?
7. What is the key assumption needed for DiD to hold (the parallel trends assumption)? If the case trends for the treated and control groups were not already parallel before the intervention, how would that distort the result?

In [ ]:
# Food poisoning 2x2 table: ate the cold cucumber salad vs did not x became ill vs did not (synthetic teaching data, not a real case)
a, b = 90, 30    # ate the suspect dish: ill / not ill
c, d = 20, 180   # did not eat the suspect dish: ill / not ill
print(f"Ate the suspect dish: {a + b} people (ill {a}, not ill {b})")
print(f"Did not eat the suspect dish: {c + d} people (ill {c}, not ill {d})")

# Sanitation-measure DiD panel: some schools received enhanced sanitation (treated); compare daily reported case counts before vs after the intervention (synthetic teaching data)
rng = np.random.default_rng(1209)
_rows = []
TRUE_EFFECT = -4.0   # enhanced sanitation measures reduce average daily reported cases by 4
for school in range(120):
    treated = 1 if school < 60 else 0
    base = rng.normal(10, 2)
    for post in (0, 1):
        cases = base - 1 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 1.5)
        _rows.append({"school": school, "treated": treated, "post": post, "cases": cases})
food = pd.DataFrame(_rows)
print(f"DiD data: {food['school'].nunique()} schools x 2 periods, true effect = {TRUE_EFFECT} cases/day")

In [ ]:
# TODO: Compute the attack rate for the groups that ate vs did not eat the suspect dish
# risk_exp = a / (a + b)
# risk_unexp = c / (c + d)

# TODO: Compute the risk ratio RR and the attributable risk AR (risk difference)
# RR = risk_exp / risk_unexp
# AR = risk_exp - risk_unexp

# TODO: Compute Pe (the proportion of everyone who ate the dish), then use the Levin formula to compute PAF (population attributable fraction)
# Pe = (a + b) / (a + b + c + d)
# PAF = Pe * (RR - 1) / (1 + Pe * (RR - 1))

# TODO: Interpret the PAF -- "the proportion of cases that could theoretically be prevented by removing this dish"

# TODO: Fit the DiD regression using the food panel data
# fit = smf.ols("cases ~ treated + post + treated:post", data=food).fit(cov_type="HC3")

# TODO: Extract the treated:post interaction coefficient and confidence interval, and compare with the true effect

# TODO: Explain the parallel trends assumption, and how the DiD estimate would be distorted if the two groups' trends were not parallel before the intervention